In [1]:
import pandas as pd
from pyprojroot import here

In [2]:
RAW_DATA_DIR = here() / "data" / "raw"
INTERIM_DATA_DIR = here() / "data" / "interim"

In [3]:
c2020 = pd.read_csv(RAW_DATA_DIR/"population/census_2020/conjunto_de_datos_iter_00CSV20.csv", low_memory=False)
c2010 = pd.read_csv(RAW_DATA_DIR/"population/census_2010/iter_00_cpv2010.csv")

In [4]:
c2020['CVEGEO'] = c2020['ENTIDAD'] * 1000 + c2020['MUN']
c2010['CVEGEO'] = c2010['entidad'] * 1000 + c2010['mun']

In [5]:
c2020_redux = c2020[['CVEGEO', 'POBTOT']].copy()
c2010_redux = c2010[['CVEGEO', 'pobtot']].copy()

In [6]:
c2020_redux.rename(columns={'POBTOT': '2020', 'anio': 'year'}, inplace=True)
c2010_redux.rename(columns={'pobtot': '2010', 'anio': 'year'}, inplace=True)

In [7]:
cdf2020 = c2020_redux.groupby('CVEGEO')[['2020']].sum()
cdf2010 = c2010_redux.groupby('CVEGEO')[['2010']].sum()

In [8]:
cdf = pd.merge(cdf2020, cdf2010, on='CVEGEO', how='outer')

In [9]:
cdf2025 = cdf.reset_index().set_index('CVEGEO').copy()
cdf2025['grw'] = (cdf2025['2020'] - cdf2025['2010'])/10
for anio in range(2011, 2020):
    cdf2025[anio] = cdf2025['2010'] + cdf2025['grw'] * (anio - 2010)

for anio in range(2021, 2026):
    cdf2025[anio] = cdf2025['2020'] + cdf2025['grw'] * (anio - 2020)

cdf2025 = cdf2025.drop(columns='grw').reset_index()

result = cdf2025.melt(id_vars='CVEGEO', var_name='year', value_name='population')

In [10]:
result.to_csv(INTERIM_DATA_DIR/"population_2015_2025B.csv")

print(f" {len(result)} municipalities processed")
print(result.sample(10))

 40032 municipalities processed
       CVEGEO  year  population
35436   12022  2024     74698.8
26498   20449  2019      6622.3
13342   16027  2014     10159.8
9430    25016  2012     44159.8
14849   31003  2014     22294.8
32777    8037  2023   3133607.5
10239    8017  2013    325822.9
10217    7121  2013         NaN
33643   20088  2023      5829.4
3602    20071  2010      6442.0
